# Mô hình Machine learning đánh giá thiết kế đồ họa

## Data source: https://huggingface.co/datasets/creative-graphic-design/GraphicDesignEvaluation

## Link Github: https://github.com/Nguyeenxdqng/NguyenHongDang24022956.git

## Web Demo:

# 0. Cài đặt thư viện cần thiết

In [1]:
import numpy as np
import torch
import io
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import pearsonr, spearmanr
import pytesseract
from transformers import LayoutLMv3Processor, LayoutLMv3Model
import joblib
import warnings
warnings.filterwarnings('ignore')

kiểm tra và lựa chọn thiết bị phần cứng

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Load Data

In [3]:
df_alignment = pd.read_parquet("hf://datasets/creative-graphic-design/GraphicDesignEvaluation/absolute-gpt-alignment/train-00000-of-00001.parquet")
df_overlap = pd.read_parquet("hf://datasets/creative-graphic-design/GraphicDesignEvaluation/absolute-gpt-overlap/train-00000-of-00001.parquet")
df_whitespace = pd.read_parquet("hf://datasets/creative-graphic-design/GraphicDesignEvaluation/absolute-gpt-whitespace/train-00000-of-00001.parquet")

tạo cột label mới và ép kiểu cho số điểm (avg) về dạng số thực

In [4]:
df_alignment['label']  = df_alignment['avg'].astype(float)
df_overlap['label']    = df_overlap['avg'].astype(float)
df_whitespace['label'] = df_whitespace['avg'].astype(float)

Tạo dataframe tổng hợp

# 2. Khởi tạo mô h LayoutLMv3

In [5]:
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
layoutlm_processor = LayoutLMv3Processor.from_pretrained(
    "microsoft/layoutlmv3-base",
    apply_ocr=True
)
layoutlm_model = LayoutLMv3Model.from_pretrained("microsoft/layoutlmv3-base").to(DEVICE)
layoutlm_model.eval()

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

LayoutLMv3Model(
  (embeddings): LayoutLMv3TextEmbeddings(
    (word_embeddings): Embedding(50265, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (x_position_embeddings): Embedding(1024, 128)
    (y_position_embeddings): Embedding(1024, 128)
    (h_position_embeddings): Embedding(1024, 128)
    (w_position_embeddings): Embedding(1024, 128)
  )
  (patch_embed): LayoutLMv3PatchEmbeddings(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (norm): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
  (encoder): LayoutLMv3Encoder(
    (layer): ModuleList(
      (0-11): 12 x LayoutLMv3Layer(
        (attention): Layo

## 3. Hàm tính đặc trưng hình học (Geometric Features)

In [6]:
def compute_geometric_features(boxes_raw: np.ndarray) -> np.ndarray:
    CANVAS = 1000.0  # LayoutLMv3 normalize bbox về 0-1000

    # Lọc bỏ padding boxes [0,0,0,0]
    valid = boxes_raw[(boxes_raw[:, 2] > boxes_raw[:, 0]) &
                      (boxes_raw[:, 3] > boxes_raw[:, 1])]

    if len(valid) < 2:
        return np.zeros(12, dtype=np.float32)

    x0, y0, x1, y1 = valid[:,0], valid[:,1], valid[:,2], valid[:,3]
    cx = (x0 + x1) / 2.0
    cy = (y0 + y1) / 2.0
    w  = (x1 - x0).clip(min=0)
    h  = (y1 - y0).clip(min=0)
    area = w * h

    # ── Alignment features ─────────────────────────────────────────
    rng_x0 = x0.max() - x0.min() + 1e-6
    rng_x1 = x1.max() - x1.min() + 1e-6
    rng_cx = cx.max() - cx.min() + 1e-6

    left_edge_std   = float(np.std(x0) / rng_x0)
    right_edge_std  = float(np.std(x1) / rng_x1)
    center_x_std    = float(np.std(cx) / rng_cx)

    # Regularity: khoảng cách dọc giữa các hàng có đều không
    sorted_cy = np.sort(np.unique(cy.round(-1)))  # làm tròn để nhóm cùng hàng
    if len(sorted_cy) >= 2:
        gaps = np.diff(sorted_cy)
        center_y_regularity = float(1.0 - np.std(gaps) / (np.mean(gaps) + 1e-6))
    else:
        center_y_regularity = 1.0

    # ── Overlap features ──────────────────────────────────────────
    n = len(valid)
    overlap_pixels = 0.0
    iou_list = []
    overlap_pairs = 0

    # Giới hạn N để tránh O(n²) quá lâu
    sample = valid[:min(n, 60)]
    ns = len(sample)
    sx0,sy0,sx1,sy1 = sample[:,0],sample[:,1],sample[:,2],sample[:,3]

    for i in range(ns):
        for j in range(i+1, ns):
            ix0 = max(sx0[i], sx0[j]);  iy0 = max(sy0[i], sy0[j])
            ix1 = min(sx1[i], sx1[j]);  iy1 = min(sy1[i], sy1[j])
            inter = max(0, ix1-ix0) * max(0, iy1-iy0)
            if inter > 0:
                a_i = (sx1[i]-sx0[i]) * (sy1[i]-sy0[i])
                a_j = (sx1[j]-sx0[j]) * (sy1[j]-sy0[j])
                union = a_i + a_j - inter
                iou_list.append(inter / (union + 1e-6))
                overlap_pixels += inter
                overlap_pairs += 1

    canvas_area = CANVAS * CANVAS
    overlap_ratio      = float(overlap_pixels / canvas_area)
    max_iou            = float(max(iou_list)) if iou_list else 0.0
    total_pairs        = ns * (ns - 1) / 2
    overlap_pair_ratio = float(overlap_pairs / (total_pairs + 1e-6))

    # ── Whitespace features ────────────────────────────────────────
    total_element_area = float(np.sum(area))
    whitespace_ratio   = float(1.0 - total_element_area / canvas_area)

    margin_left  = float(x0.min() / CANVAS)
    margin_right = float((CANVAS - x1.max()) / CANVAS)
    margin_top   = float(y0.min() / CANVAS)

    # Density grid 4×4
    grid_n = 4
    density = np.zeros((grid_n, grid_n))
    for k in range(len(valid)):
        col = min(int(cx[k] / CANVAS * grid_n), grid_n - 1)
        row = min(int(cy[k] / CANVAS * grid_n), grid_n - 1)
        density[row, col] += area[k] / canvas_area
    density_variance = float(np.var(density))

    feat = np.array([
        left_edge_std,       # [0]  alignment
        right_edge_std,      # [1]  alignment
        center_x_std,        # [2]  alignment
        center_y_regularity, # [3]  alignment
        overlap_ratio,       # [4]  overlap
        max_iou,             # [5]  overlap
        overlap_pair_ratio,  # [6]  overlap
        whitespace_ratio,    # [7]  whitespace
        margin_left,         # [8]  whitespace
        margin_right,        # [9]  whitespace
        margin_top,          # [10] whitespace
        density_variance,    # [11] whitespace
    ], dtype=np.float32)

    return feat

Định nghĩa các đặc trưng

In [7]:
# Tên các đặc trưng để giải thích kết quả sau này
GEO_FEATURE_NAMES = [
    "geo_left_edge_std",                 #Độ lệch chuẩn cạnh trái → nhỏ = các phần tử căn đều bên trái
    "geo_right_edge_std",                #Độ lệch chuẩn cạnh phải → nhỏ = căn phải đều
    "geo_center_x_std",                  #Độ lệch chuẩn tâm ngang → nhỏ = canh giữa đều
    "geo_center_y_regularity",           #Khoảng cách hàng đều nhau → lớn = nhịp điệu đọc tốt
    "geo_overlap_ratio",                 #Tỉ lệ pixel chồng lấp/canvas → gần 0 = không bị che khuất
    "geo_max_iou",                       #IoU lớn nhất giữa 2 phần tử → gần 0 = không có cặp nào chồng
    "geo_overlap_pair_ratio",            # % cặp phần tử có chồng lấp → nhỏ = thiết kế gọn gàng
    "geo_whitespace_ratio",              #Tỉ lệ diện tích trống → ~0.4–0.6 là tối ưu
    "geo_margin_left",                   #Lề trái nhỏ nhất → >0.05 = không bị sát biên
    "geo_margin_right",                  #Lề phải nhỏ nhất → cân đối với margin_left
    "geo_margin_top",                    #Lề trên nhỏ nhất → không gian thở ở đầu
    "geo_density_variance",              #Variance mật độ lưới 4×4 → nhỏ = phân bố đều
]

## 4. Hàm trích xuất đặc trưng lai (Hybrid Feature Extraction)

In [8]:
def extract_hybrid_features(df, batch_size=8, label_col='label'):

    all_feat   = []
    all_labels = []
    n = len(df)

    for start in range(0, n, batch_size):
        end   = min(start + batch_size, n)
        batch = df.iloc[start:end]

        images = []
        for img in batch["image"]:
            if isinstance(img, dict):
                if img.get('bytes'):
                    img = Image.open(io.BytesIO(img['bytes']))
                elif img.get('path'):
                    img = Image.open(img['path'])
            elif type(img).__name__ == "ndarray":
                img = Image.fromarray(img)
            if img.mode != "RGB":
                img = img.convert("RGB")
            images.append(img)

        inputs = layoutlm_processor(
            images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(DEVICE)

        with torch.no_grad():
            outputs = layoutlm_model(**inputs)

        hidden = outputs.last_hidden_state  # [B, seq_len, 768]
        B, seq_len, D = hidden.shape

        # LayoutLMv3: text tokens chiếm seq_len-196 đầu, 196 patch tokens ở cuối
        N_PATCHES = 196
        text_end  = seq_len - N_PATCHES  # vị trí bắt đầu patch tokens

        for i in range(len(images)):
            # ── Phần 1: CLS token [0] ──────────────────────────────
            cls_vec = hidden[i, 0, :].cpu().float().numpy()
            cls_vec = cls_vec / (np.linalg.norm(cls_vec) + 1e-8)

            # ── Phần 2: Visual patch tokens ──────────────────────
            if text_end < seq_len:
                patch_tokens = hidden[i, text_end:, :].cpu().float().numpy()
            else:
                # Fallback: dùng tất cả tokens ngoài CLS
                patch_tokens = hidden[i, 1:, :].cpu().float().numpy()

            vis_mean = patch_tokens.mean(axis=0)
            vis_std  = patch_tokens.std(axis=0)
            vis_mean = vis_mean / (np.linalg.norm(vis_mean) + 1e-8)
            vis_std  = vis_std  / (np.linalg.norm(vis_std)  + 1e-8)
            visual_vec = np.concatenate([vis_mean, vis_std])  # 1536d

            # ── Phần 3: Geometric features từ OCR bboxes ─────────
            # inputs['bbox'] shape: [B, seq_len, 4] — toạ độ x0,y0,x1,y1 (0-1000)
            boxes = inputs['bbox'][i].cpu().numpy()           # [seq_len, 4]
            mask  = inputs['attention_mask'][i].cpu().numpy().astype(bool)
            valid_boxes = boxes[mask]                          # lọc padding
            geo_vec = compute_geometric_features(valid_boxes)  # 12d

            # ── Ghép lại ──────────────────────────────────────────
            feat = np.concatenate([cls_vec, visual_vec, geo_vec])
            all_feat.append(feat)

        labels = batch[label_col].values
        all_labels.extend(labels)


    X = np.vstack(all_feat)
    y = np.array(all_labels, dtype=np.float32)

    return X, y


## 5. Trích xuất đặc trưng alignment + whitespace + overlap

In [9]:
X_alignment, y_alignment = extract_hybrid_features(df_alignment, batch_size=16)
X_whitespace, y_whitespace = extract_hybrid_features(df_whitespace, batch_size=16)
X_overlap, y_overlap = extract_hybrid_features(df_overlap, batch_size=16)

# 6. Chia tập dữ liệu và chuẩn hóa

## 6.1 Alignment chia tập dữ liệu train/test +  chuẩn hóa + giảm chiều dữ liệu PCA

In [10]:
X_train_alignment, X_test_alignment, y_train_alignment, y_test_alignment = train_test_split(
    X_alignment, y_alignment, test_size=0.2, random_state=42
)

In [11]:
scaler = StandardScaler()
X_train_sc_alignment = scaler.fit_transform(X_train_alignment)
X_test_sc_alignment  = scaler.transform(X_test_alignment)

pca = PCA(n_components=100, random_state=42)
X_train_pca_alignment   = pca.fit_transform(X_train_sc_alignment[:, :-12])  # ✅ -12 thay vì :1536
X_test_pca_alignment    = pca.transform(X_test_sc_alignment[:, :-12])

X_train_final_alignment = np.hstack([X_train_pca_alignment, X_train_sc_alignment[:, -12:]])  # 100+12 = 112
X_test_final_alignment  = np.hstack([X_test_pca_alignment,  X_test_sc_alignment[:,  -12:]])

## 6.2 Whitespace - chia tập dữ liệu train/test +  chuẩn hóa + giảm chiều dữ liệu PCA

In [12]:
X_train_whitespace, X_test_whitespace, y_train_whitespace, y_test_whitespace = train_test_split(
    X_whitespace, y_whitespace, test_size=0.2, random_state=42
)

In [13]:
scaler = StandardScaler()
X_train_sc_whitespace = scaler.fit_transform(X_train_whitespace)
X_test_sc_whitespace  = scaler.transform(X_test_whitespace)

pca = PCA(n_components=100, random_state=42)
X_train_pca_whitespace   = pca.fit_transform(X_train_sc_whitespace[:, :-12])  # ✅ -12 thay vì :1536
X_test_pca_whitespace    = pca.transform(X_test_sc_whitespace[:, :-12])

X_train_final_whitespace = np.hstack([X_train_pca_whitespace, X_train_sc_whitespace[:, -12:]])  # 100+12 = 112
X_test_final_whitespace  = np.hstack([X_test_pca_whitespace,  X_test_sc_whitespace[:,  -12:]])

## 6.3 Overlap - chia tập dữ liệu train/test +  chuẩn hóa + giảm chiều dữ liệu PCA

In [14]:
X_train_overlap, X_test_overlap, y_train_overlap, y_test_overlap = train_test_split(
    X_overlap, y_overlap, test_size=0.2, random_state=42
)

In [15]:
scaler = StandardScaler()
X_train_sc_overlap = scaler.fit_transform(X_train_overlap)
X_test_sc_overlap  = scaler.transform(X_test_overlap)

pca = PCA(n_components=100, random_state=42)
X_train_pca_overlap   = pca.fit_transform(X_train_sc_overlap[:, :-12])  # ✅ -12 thay vì :1536
X_test_pca_overlap    = pca.transform(X_test_sc_overlap[:, :-12])

X_train_final_overlap = np.hstack([X_train_pca_overlap, X_train_sc_overlap[:, -12:]])  # 100+12 = 112
X_test_final_overlap  = np.hstack([X_test_pca_overlap,  X_test_sc_overlap[:,  -12:]])

# 7. Hàm đánh giá mô hình

In [25]:
def evaluate_model(model, X_tr, X_te, y_tr, y_te, model_name):
    print(f"{'=' * 55}")
    print(f"🤖 MÔ HÌNH: {model_name}")
    print(f"{'=' * 55}")

    # Huấn luyện mô hình
    model.fit(X_tr, y_tr)

    y_pred_tr = model.predict(X_tr)
    y_pred_te = model.predict(X_te)

    # Clip dữ liệu về khoảng 0-10
    y_pred_tr = np.clip(y_pred_tr, 0, 10)
    y_pred_te = np.clip(y_pred_te, 0, 10)

    # Chỉ số hồi quy cơ bản (Sklearn tự động lấy trung bình cho đa đầu ra)
    metrics = {
        "Train MSE": mean_squared_error(y_tr, y_pred_tr),
        "Test  MSE": mean_squared_error(y_te, y_pred_te),
        "Train MAE": mean_absolute_error(y_tr, y_pred_tr),
        "Test  MAE": mean_absolute_error(y_te, y_pred_te),
        "Train R²": r2_score(y_tr, y_pred_tr),
        "Test  R²": r2_score(y_te, y_pred_te),
    }

    # Chuyển đổi dữ liệu sang mảng numpy để xử lý số chiều
    y_te_arr = np.asarray(y_te)
    y_pred_te_arr = np.asarray(y_pred_te)

    # Tính Pearson & Spearman (Tự động tính theo từng cột rồi lấy trung bình nếu là đa đầu ra)
    if y_te_arr.ndim > 1 and y_te_arr.shape[1] > 1:
        p_list = [pearsonr(y_te_arr[:, i], y_pred_te_arr[:, i])[0] for i in range(y_te_arr.shape[1])]
        s_list = [spearmanr(y_te_arr[:, i], y_pred_te_arr[:, i])[0] for i in range(y_te_arr.shape[1])]
        pearson_r = np.mean(p_list)
        spearman_r = np.mean(s_list)
    else:
        pearson_r, _ = pearsonr(y_te_arr.ravel(), y_pred_te_arr.ravel())
        spearman_r, _ = spearmanr(y_te_arr.ravel(), y_pred_te_arr.ravel())

    metrics["Pearson r"] = pearson_r
    metrics["Spearman ρ"] = spearman_r

    # In các chỉ số (An toàn tuyệt đối không lo lỗi định dạng)
    for k, v in metrics.items():
        print(f"   {k:<14}: {v:>8.4f}")

    # Cảnh báo phân phối dự đoán
    pred_std = y_pred_te_arr.std()
    true_std = y_te_arr.std()
    print(f"\n   Std dự đoán  : {pred_std:.4f}  (thực tế: {true_std:.4f})")
    if pred_std < true_std * 0.4:
        print("   ⚠️  Dự đoán bị co cụm quá nhiều — model vẫn thiên về mean.")
    else:
        print("   ✅ Dự đoán có đủ độ phân tán.")

    # Đánh giá chỉ số R²
    r2 = metrics["Test  R²"]
    if r2 > 0.5:
        print(f"   ✅ R²={r2:.3f} — Model giải thích được >50% variance")
    elif r2 > 0.2:
        print(f"   🟡 R²={r2:.3f} — Model có học được nhưng còn yếu")
    else:
        print(f"   🔴 R²={r2:.3f} — Model hầu như không học được pattern")

    # Ví dụ dự đoán hiển thị trực quan cho cả đơn/đa đầu ra
    print(f"\n     10 mẫu test đầu tiên:")
    print(f"   {'Thực tế':>15} {'Dự đoán':>15} {'Sai lệch':>15}")
    print(f"   {'-' * 49}")

    for a, p in zip(y_te_arr[:10], y_pred_te_arr[:10]):
        if y_te_arr.ndim > 1 and y_te_arr.shape[1] > 1:
            a_str = f"[{','.join([f'{x:.2f}' for x in a])}]"
            p_str = f"[{','.join([f'{x:.2f}' for x in p])}]"
            diff_str = f"[{','.join([f'{x-y:+.2f}' for x, y in zip(p, a)])}]"
            print(f"   {a_str:>15} {p_str:>15} {diff_str:>15}")
        else:
            print(f"   {a:>15.4f} {p:>15.4f} {p - a:>+15.4f}")

    print()
    return model, metrics

# 8. Huấn luyện và so sánh các mô hình

## 8.1 Alignment - Huấn luyện và so sánh các mô hình

In [30]:
results_alignment = {}

# ── Mô hình 1: GradientBoosting (mạnh hơn RF với features chiều cao) ──
gb_model_alignment, gb_metrics_alignment = evaluate_model(
    GradientBoostingRegressor(
        n_estimators=300,
        max_depth=7,           # sâu hơn RF để học pattern phức tạp
        learning_rate=0.02,
        subsample=0.8,
        random_state=42
    ),
    X_train_final_alignment, X_test_final_alignment, y_train_alignment, y_test_alignment,
    "Gradient Boosting Regressor"
)
results_alignment["GradientBoosting"] = gb_metrics_alignment

# ── Mô hình 2: Random Forest (baseline, cải thiện max_depth) ──────────
rf_model_alignment, rf_metrics_alignment = evaluate_model(
    RandomForestRegressor(
        n_estimators=200,
        max_depth=8,           # tăng từ 2 → 8 để model học được nhiều hơn
        min_samples_leaf=4,
        random_state=42,
        n_jobs=-1
    ),
    X_train_final_alignment, X_test_final_alignment, y_train_alignment, y_test_alignment,
    "Random Forest (max_depth=8)"
)
results_alignment["RandomForest"] = rf_metrics_alignment

# ── Mô hình 3: SVR ─────────────────────────────────────────────────────
svr_model_alignment, svr_metrics_alignment = evaluate_model(
    SVR(kernel="rbf", C=8, epsilon=0.05, gamma="scale"),
    X_train_final_alignment, X_test_final_alignment, y_train_alignment, y_test_alignment,
    "SVR (RBF, C=5)"
)
results_alignment["SVR"] = svr_metrics_alignment

# Tổng kết
print("\n" + "="*55)
print("📊 SO SÁNH CÁC MÔ HÌNH (Test set)")
print("="*55)
print(f"{'Model':<25} {'R²':>8} {'MAE':>8} {'Pearson':>10}")
print("-"*55)
for name, m in results_alignment.items():
    print(f"{name:<25} {m['Test  R²']:>8.4f} {m['Test  MAE']:>8.4f} {m['Pearson r']:>10.4f}")


🤖 MÔ HÌNH: Gradient Boosting Regressor
   Train MSE     :   0.0028
   Test  MSE     :   0.9998
   Train MAE     :   0.0445
   Test  MAE     :   0.8260
   Train R²      :   0.9987
   Test  R²      :   0.3545
   Pearson r     :   0.6378
   Spearman ρ    :   0.6635

   Std dự đoán  : 0.8971  (thực tế: 1.2445)
   ✅ Dự đoán có đủ độ phân tán.
   🟡 R²=0.354 — Model có học được nhưng còn yếu

     10 mẫu test đầu tiên:
           Thực tế         Dự đoán        Sai lệch
   -------------------------------------------------
            7.4000          6.2191         -1.1809
            7.0000          6.8243         -0.1757
            4.8000          5.6230         +0.8230
            6.8000          6.4892         -0.3108
            6.6000          6.1771         -0.4229
            6.2000          5.2553         -0.9447
            8.6000          8.5259         -0.0741
            5.6000          6.0139         +0.4139
            7.0000          6.8349         -0.1651
            7.4000   

## 8.2 whitespace - Huấn luyện và so sánh các mô hình

In [31]:
results_whitespace = {}

# ── Mô hình 1: GradientBoosting (mạnh hơn RF với features chiều cao) ──
gb_model_whitespace, gb_metrics_whitespace = evaluate_model(
    GradientBoostingRegressor(
        n_estimators=300,
        max_depth=3,           # sâu hơn RF để học pattern phức tạp
        learning_rate=0.02,
        subsample=0.8,
        random_state=42
    ),
    X_train_final_whitespace, X_test_final_whitespace, y_train_whitespace, y_test_whitespace,
    "Gradient Boosting Regressor"
)
results_whitespace["GradientBoosting"] = gb_metrics_whitespace

# ── Mô hình 2: Random Forest (baseline, cải thiện max_depth) ──────────
rf_model_whitespace, rf_metrics_whitespace = evaluate_model(
    RandomForestRegressor(
        n_estimators=200,
        max_depth=7,           # tăng từ 2 → 8 để model học được nhiều hơn
        min_samples_leaf=4,
        random_state=42,
        n_jobs=-1
    ),
    X_train_final_whitespace, X_test_final_whitespace, y_train_whitespace, y_test_whitespace,
    "Random Forest (max_depth=8)"
)
results_whitespace["RandomForest"] = rf_metrics_whitespace

# ── Mô hình 3: SVR ─────────────────────────────────────────────────────
svr_model_whitespace, svr_metrics_whitespace = evaluate_model(
    SVR(kernel="rbf", C=5, epsilon=0.05, gamma="scale"),
    X_train_final_whitespace, X_test_final_whitespace, y_train_whitespace, y_test_whitespace,
    "SVR (RBF, C=5)"
)
results_whitespace["SVR"] = svr_metrics_whitespace

# Tổng kết
print("\n" + "="*55)
print("📊 SO SÁNH CÁC MÔ HÌNH (Test set)")
print("="*55)
print(f"{'Model':<25} {'R²':>8} {'MAE':>8} {'Pearson':>10}")
print("-"*55)
for name, m in results_whitespace.items():
    print(f"{name:<25} {m['Test  R²']:>8.4f} {m['Test  MAE']:>8.4f} {m['Pearson r']:>10.4f}")


🤖 MÔ HÌNH: Gradient Boosting Regressor
   Train MSE     :   0.1690
   Test  MSE     :   1.2451
   Train MAE     :   0.3420
   Test  MAE     :   0.8577
   Train R²      :   0.9198
   Test  R²      :   0.3691
   Pearson r     :   0.6157
   Spearman ρ    :   0.6199

   Std dự đoán  : 0.7702  (thực tế: 1.4048)
   ✅ Dự đoán có đủ độ phân tán.
   🟡 R²=0.369 — Model có học được nhưng còn yếu

     10 mẫu test đầu tiên:
           Thực tế         Dự đoán        Sai lệch
   -------------------------------------------------
            7.2000          5.8149         -1.3851
            7.2000          5.5540         -1.6460
            5.4000          5.0522         -0.3478
            5.2000          5.7009         +0.5009
            6.8000          5.2135         -1.5865
            5.2000          5.0782         -0.1218
            7.8000          7.1876         -0.6124
            6.0000          4.5888         -1.4112
            5.0000          4.9721         -0.0279
            6.2000   

## 8.3 Overlap - Huấn luyện và so sánh các mô hình

In [32]:
results_overlap = {}

# ── Mô hình 1: GradientBoosting (mạnh hơn RF với features chiều cao) ──
gb_model_overlap, gb_metrics_overlap = evaluate_model(
    GradientBoostingRegressor(
        n_estimators=300,
        max_depth=3,           # sâu hơn RF để học pattern phức tạp
        learning_rate=0.02,
        subsample=0.8,
        random_state=42
    ),
    X_train_final_overlap, X_test_final_overlap, y_train_overlap, y_test_overlap,
    "Gradient Boosting Regressor"
)
results_overlap["GradientBoosting"] = gb_metrics_overlap

# ── Mô hình 2: Random Forest (baseline, cải thiện max_depth) ──────────
rf_model_overlap, rf_metrics_overlap = evaluate_model(
    RandomForestRegressor(
        n_estimators=200,
        max_depth=7,           # tăng từ 2 → 8 để model học được nhiều hơn
        min_samples_leaf=4,
        random_state=42,
        n_jobs=-1
    ),
    X_train_final_overlap, X_test_final_overlap, y_train_overlap, y_test_overlap,
    "Random Forest (max_depth=8)"
)
results_overlap["RandomForest"] = rf_metrics_overlap

# ── Mô hình 3: SVR ─────────────────────────────────────────────────────
svr_model_overlap, svr_metrics_overlap = evaluate_model(
    SVR(kernel="rbf", C=5, epsilon=0.05, gamma="scale"),
    X_train_final_overlap, X_test_final_overlap, y_train_overlap, y_test_overlap,
    "SVR (RBF, C=5)"
)
results_overlap["SVR"] = svr_metrics_overlap

# Tổng kết
print("\n" + "="*55)
print("📊 SO SÁNH CÁC MÔ HÌNH (Test set)")
print("="*55)
print(f"{'Model':<25} {'R²':>8} {'MAE':>8} {'Pearson':>10}")
print("-"*55)
for name, m in results_overlap.items():
    print(f"{name:<25} {m['Test  R²']:>8.4f} {m['Test  MAE']:>8.4f} {m['Pearson r']:>10.4f}")


🤖 MÔ HÌNH: Gradient Boosting Regressor
   Train MSE     :   0.2526
   Test  MSE     :   1.6883
   Train MAE     :   0.4219
   Test  MAE     :   0.9994
   Train R²      :   0.9178
   Test  R²      :   0.4231
   Pearson r     :   0.6612
   Spearman ρ    :   0.6542

   Std dự đoán  : 0.9283  (thực tế: 1.7107)
   ✅ Dự đoán có đủ độ phân tán.
   🟡 R²=0.423 — Model có học được nhưng còn yếu

     10 mẫu test đầu tiên:
           Thực tế         Dự đoán        Sai lệch
   -------------------------------------------------
            7.6000          5.6934         -1.9066
            6.6000          4.9885         -1.6115
            4.0000          4.6681         +0.6681
            4.6000          5.3609         +0.7609
            7.4000          5.4384         -1.9616
            6.0000          5.8971         -0.1029
            8.2000          6.9704         -1.2296
            4.8000          4.3780         -0.4220
            4.8000          4.8337         +0.0337
            4.6000   

## 9. Lưu model và hàm chấm điểm ảnh mới

In [34]:
joblib.dump(gb_model_alignment,  "gb_model_alignment.pkl")
joblib.dump(rf_model_alignment,  "rf_model_alignment.pkl")
joblib.dump(svr_model_alignment, "svr_model_alignment.pkl")
joblib.dump(gb_model_whitespace,  "gb_model_whitespace.pkl")
joblib.dump(rf_model_whitespace,  "rf_model_whitespace.pkl")
joblib.dump(svr_model_whitespace, "svr_model_whitespace.pkl")
joblib.dump(gb_model_overlap,  "gb_model_overlap.pkl")
joblib.dump(rf_model_overlap,  "rf_model_overlap.pkl")
joblib.dump(svr_model_overlap, "svr_model_overlap.pkl")
joblib.dump(scaler,    "scaler.pkl")
joblib.dump(pca,       "pca.pkl")

['pca.pkl']

Re-fitting best models cho từng tiêu chí

In [21]:
trained_alignment  = {}
trained_whitespace = {}
trained_overlap    = {}

for name, (ModelClass, kwargs) in [
    ("GradientBoosting", (GradientBoostingRegressor, dict(n_estimators=300, max_depth=3, learning_rate=0.02, subsample=0.8, random_state=42))),
    ("RandomForest",     (RandomForestRegressor,     dict(n_estimators=200, max_depth=7, min_samples_leaf=4, random_state=42, n_jobs=-1))),
    ("SVR",              (SVR,                        dict(kernel="rbf", C=5, epsilon=0.05, gamma="scale"))),
]:
    m_a = ModelClass(**kwargs); m_a.fit(X_train_final_alignment,  y_train_alignment)
    m_w = ModelClass(**kwargs); m_w.fit(X_train_final_whitespace, y_train_whitespace)
    m_o = ModelClass(**kwargs); m_o.fit(X_train_final_overlap,    y_train_overlap)
    trained_alignment[name]  = m_a
    trained_whitespace[name] = m_w
    trained_overlap[name]    = m_o



## 10. Hàm chấm điểm thiết kế mới

In [22]:
def score_design(image_source, weights=None):

    if weights is None:
        weights = {"alignment": 1/3, "whitespace": 1/3, "overlap": 1/3}
    assert abs(sum(weights.values()) - 1.0) < 1e-6, "Tổng trọng số phải = 1"

    # ── Load ảnh ──
    if isinstance(image_source, str):
        if image_source.startswith("http"):
            import urllib.request
            with urllib.request.urlopen(image_source) as r:
                img = Image.open(io.BytesIO(r.read()))
        else:
            img = Image.open(image_source)
    elif isinstance(image_source, Image.Image):
        img = image_source
    else:
        raise TypeError("Cần đường dẫn, URL hoặc PIL Image")

    if img.mode != "RGB":
        img = img.convert("RGB")
    w, h = img.size
    if w < 224 or h < 224:
        scale = max(224/w, 224/h)
        img = img.resize((int(w*scale), int(h*scale)), Image.LANCZOS)

    print(f"🖼️  Kích thước: {img.size} | Mode: {img.mode}")
    print("⏳ Đang trích xuất hybrid features...")

    # ── Trích xuất feature ──
    inputs = layoutlm_processor(
        img, return_tensors="pt",
        padding=True, truncation=True, max_length=512
    ).to(DEVICE)

    with torch.no_grad():
        outputs = layoutlm_model(**inputs)

    hidden   = outputs.last_hidden_state
    seq_len  = hidden.shape[1]
    text_end = seq_len - 196

    cls_vec = hidden[0, 0, :].cpu().float().numpy()
    cls_vec = cls_vec / (np.linalg.norm(cls_vec) + 1e-8)

    patch_tokens = hidden[0, text_end:, :].cpu().float().numpy()
    vis_mean = patch_tokens.mean(axis=0); vis_mean /= (np.linalg.norm(vis_mean) + 1e-8)
    vis_std  = patch_tokens.std(axis=0);  vis_std  /= (np.linalg.norm(vis_std)  + 1e-8)

    boxes   = inputs['bbox'][0].cpu().numpy()
    mask    = inputs['attention_mask'][0].cpu().numpy().astype(bool)
    geo_vec = compute_geometric_features(boxes[mask])

    feat_raw = np.concatenate([cls_vec, vis_mean, vis_std, geo_vec]).reshape(1, -1)

    # ── Dự đoán từng tiêu chí (Ensemble 3 bộ mô hình riêng) ──
    feat_sc  = scaler.transform(feat_raw)
    feat_pca = pca.transform(feat_sc[:, :-12])
    feat_f   = np.hstack([feat_pca, feat_sc[:, -12:]])

    # Dự đoán alignment, whitespace, overlap bằng GradientBoosting (best model)
    # Nếu muốn dùng RF/SVR, thay trained_alignment["RandomForest"] v.v.
    score_alignment  = float(np.clip(trained_alignment["GradientBoosting"].predict(feat_f)[0],  0, 10))
    score_whitespace = float(np.clip(trained_whitespace["GradientBoosting"].predict(feat_f)[0], 0, 10))
    score_overlap    = float(np.clip(trained_overlap["GradientBoosting"].predict(feat_f)[0],    0, 10))

    # Overall = trung bình có trọng số
    score_overall = round(
        weights["alignment"]  * score_alignment +
        weights["whitespace"] * score_whitespace +
        weights["overlap"]    * score_overlap,
        2
    )

    verdict_map = [(8,"🟢 Xuất sắc"), (6.5,"🟡 Tốt"), (5,"🟠 Trung bình"), (0,"🔴 Cần cải thiện")]
    verdict = next(v for t, v in verdict_map if score_overall >= t)

    # ── In kết quả ──
    print("\n" + "="*55)
    print("📐 KẾT QUẢ CHẤM ĐIỂM TỔNG HỢP")
    print("="*55)
    print(f"   {'Tiêu chí':<22} {'Điểm':>6}  {'Trọng số':>10}")
    print(f"   {'-'*42}")
    print(f"   {'Alignment':<22} {score_alignment:>6.2f}  {weights['alignment']:>10.0%}")
    print(f"   {'Whitespace':<22} {score_whitespace:>6.2f}  {weights['whitespace']:>10.0%}")
    print(f"   {'Overlap':<22} {score_overlap:>6.2f}  {weights['overlap']:>10.0%}")
    print(f"   {'─'*42}")
    print(f"   {'OVERALL (Ensemble)':<22} {score_overall:>6.2f}  ◀ điểm tổng hợp")
    print(f"\n   Đánh giá: {verdict}")

    print("\n🔍 Giải thích Geometric Features:")
    geo_labels = {
        "geo_left_edge_std":       ("Alignment",  "Cạnh trái lệch nhiều = bố cục kém căn chỉnh"),
        "geo_center_x_std":        ("Alignment",  "Tâm ngang lệch = thiếu trục căn chỉnh"),
        "geo_center_y_regularity": ("Alignment",  "Khoảng cách hàng đều = nhịp điệu tốt"),
        "geo_overlap_ratio":       ("Overlap",    "Tỉ lệ chồng lấp pixel"),
        "geo_max_iou":             ("Overlap",    "Cặp phần tử chồng lấp nặng nhất"),
        "geo_whitespace_ratio":    ("Whitespace", "Tỉ lệ không gian trắng"),
        "geo_margin_left":         ("Whitespace", "Lề trái"),
        "geo_density_variance":    ("Whitespace", "Phân bố mật độ đều hay không"),
    }
    for i, name in enumerate(GEO_FEATURE_NAMES):
        val = geo_vec[i]
        if name in geo_labels:
            group, desc = geo_labels[name]
            print(f"   [{group:<10}] {name:<28}: {val:.4f}  ({desc})")

    return {
        "score_alignment":        score_alignment,
        "score_whitespace":       score_whitespace,
        "score_overlap":          score_overlap,
        "score_overall_ensemble": score_overall,
        "verdict":                verdict,
        "weights_used":           weights,
        "geo_features":           dict(zip(GEO_FEATURE_NAMES, geo_vec)),
    }


# ── Sử dụng ──

In [24]:
result = score_design("biểu đồ/sample_design_scored.png")

🖼️  Kích thước: (8159, 3105) | Mode: RGB
⏳ Đang trích xuất hybrid features...

📐 KẾT QUẢ CHẤM ĐIỂM TỔNG HỢP
   Tiêu chí                 Điểm    Trọng số
   ------------------------------------------
   Alignment                6.46         33%
   Whitespace               6.68         33%
   Overlap                  6.76         33%
   ──────────────────────────────────────────
   OVERALL (Ensemble)       6.64  ◀ điểm tổng hợp

   Đánh giá: 🟡 Tốt

🔍 Giải thích Geometric Features:
   [Alignment ] geo_left_edge_std           : 0.3039  (Cạnh trái lệch nhiều = bố cục kém căn chỉnh)
   [Alignment ] geo_center_x_std            : 0.3131  (Tâm ngang lệch = thiếu trục căn chỉnh)
   [Alignment ] geo_center_y_regularity     : 0.2978  (Khoảng cách hàng đều = nhịp điệu tốt)
   [Overlap   ] geo_overlap_ratio           : 0.0431  (Tỉ lệ chồng lấp pixel)
   [Overlap   ] geo_max_iou                 : 1.0000  (Cặp phần tử chồng lấp nặng nhất)
   [Whitespace] geo_whitespace_ratio        : 0.9533  (Tỉ lệ kh